# GIRA - Bicicletas de Lisboa

### Note on file names

Before running this notebook they were renamed to short, uniform names, which avoids read errors and keeps the loading cell readable. **The file contents were not modified in any way** only the file names on disk.

| Original download name                      | Renamed to           | Content          |
|---------------------------------------------|----------------------|------------------|
| `pdapgilgira1t2020.xlsx`                    | `gira_2020_q1.xlsx`  | 2020, quarter 1  |
| `pdapgilgira2t2020.xlsx`                    | `gira_2020_q2.xlsx`  | 2020, quarter 2  |
| `pdapgilgira3t2020.xlsx`                    | `gira_2020_q3.xlsx`  | 2020, quarter 3  |
| `pdapgilgira4t2020.xlsx`                    | `gira_2020_q4.xlsx`  | 2020, quarter 4  |
| `Gira - Bicicletas de Lisboa 2021.csv`      | `gira_2021.csv`      | 2021, full year  |
| `Gira - Bicicletas de Lisboa 1semestre 2022.csv` | `gira_2022_s1.csv` | 2022, semester 1 |
| `estacoes-gira-2--semestre-2022.csv`        | `gira_2022_s2.csv`   | 2022, semester 2 |
| `estacoes-gira-1-trimestre-2023.xlsx`       | `gira_2023_q1.xlsx`  | 2023, quarter 1  |

Naming convention: `gira_<year>_<period>.<ext>`, where the period is `q1`–`q4`
for quarters and `s1`/`s2` for semesters, matching how each file was published.
The original names mix Portuguese and English, spaces, and hyphens (including a
double hyphen in the 2022 S2 file), which is why the rename was done first.

Two things to check before running the next cell:
1. All 8 files sit in the **same folder as this notebook**, otherwise the
   relative paths will fail.
2. The 2020 and 2023 Excel files contain a second, empty sheet (`Folha1`).
   That is why every `read_excel` call uses `sheet_name=0`: it reads only the
   first sheet, where the data actually is.


## Step 1 - Getting to Know the Data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

In [2]:
# 2020 — Excel, first sheet only
gira_2020_q1 = pd.read_excel("gira_2020_q1.xlsx", sheet_name=0)
gira_2020_q2 = pd.read_excel("gira_2020_q2.xlsx", sheet_name=0)
gira_2020_q3 = pd.read_excel("gira_2020_q3.xlsx", sheet_name=0)
gira_2020_q4 = pd.read_excel("gira_2020_q4.xlsx", sheet_name=0)

# 2021 — CSV, full year
gira_2021 = pd.read_csv("gira_2021.csv")

# 2022 — CSV, two semesters
gira_2022_s1 = pd.read_csv("gira_2022_s1.csv")
gira_2022_s2 = pd.read_csv("gira_2022_s2.csv")

# 2023 — Excel, Q1 only
gira_2023_q1 = pd.read_excel("gira_2023_q1.xlsx", sheet_name=0)

In [3]:
for name, df in [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
                 ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4),
                 ("2021", gira_2021), ("2022_s1", gira_2022_s1),
                 ("2022_s2", gira_2022_s2), ("2023_q1", gira_2023_q1)]:
    print(f"{name:10} rows={len(df):>8}  cols={list(df.columns)}")

2020_q1    rows=  100000  cols=['desigcomercial', 'numbicicletas', 'numbicicletas__1', 'numdocasvacias', 'position', 'entity_ts']
2020_q2    rows=  100000  cols=['desigcomercial', 'numbicicletas', 'numbicicletas__1', 'numdocasvacias', 'position', 'entity_ts']
2020_q3    rows=  100000  cols=['desigcomercial', 'numbicicletas', 'numbicicletas__1', 'numdocasvacias', 'position', 'entity_ts']
2020_q4    rows=  100000  cols=['desigcomercial', 'numbicicletas', 'numbicicletas__1', 'numdocasvacias', 'position', 'entity_ts']
2021       rows= 2218606  cols=['desigcomercial', 'numbicicletas', 'numdocas', 'numdocasvacias', 'position', 'entity_ts', 'estado']
2022_s1    rows= 1555396  cols=['desigcomercial', 'numbicicletas', 'numdocas', 'position', 'entity_ts', 'estado']
2022_s2    rows= 2382896  cols=['desigcomercial', 'numbicicletas', 'numdocas', 'position', 'entity_ts', 'estado']
2023_q1    rows=  933644  cols=['desigcomercial', 'numbicicletas', 'numdocas', 'position', 'entity_ts', 'estado']


In [4]:
files = [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
         ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4),
         ("2021", gira_2021), ("2022_s1", gira_2022_s1),
         ("2022_s2", gira_2022_s2), ("2023_q1", gira_2023_q1)]

for name, df in files:
    print(f"=== {name} ===")
    print(f"  rows: {len(df):>8}")
    print(f"  dates: {df['entity_ts'].min()}  ->  {df['entity_ts'].max()}")
    print(f"  missing:\n{df.isna().sum().to_string().rjust(4)}")
    print()

=== 2020_q1 ===
  rows:   100000
  dates: 2020-01-01 00:04:01  ->  2020-03-31 23:47:03
  missing:
desigcomercial      0
numbicicletas       0
numbicicletas__1    0
numdocasvacias      0
position            0
entity_ts           0

=== 2020_q2 ===
  rows:   100000
  dates: 2020-04-01 00:06:04  ->  2020-06-30 23:47:06
  missing:
desigcomercial      0
numbicicletas       0
numbicicletas__1    0
numdocasvacias      0
position            0
entity_ts           0

=== 2020_q3 ===
  rows:   100000
  dates: 2020-06-30 00:06:06  ->  2020-09-30 23:56:09
  missing:
desigcomercial      0
numbicicletas       0
numbicicletas__1    0
numdocasvacias      0
position            0
entity_ts           0

=== 2020_q4 ===
  rows:   100000
  dates: 2020-10-01 00:16:10  ->  2020-12-31 23:40:12
  missing:
desigcomercial      0
numbicicletas       0
numbicicletas__1    0
numdocasvacias      0
position            0
entity_ts           0

=== 2021 ===
  rows:  2218606
  dates: 2021-01-01T12:00:23.945Z  ->  2021-12

In [5]:
gira_2020_q1.info()
print("-" * 60)
gira_2021.info()      
print("-" * 60)
gira_2023_q1.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   desigcomercial    100000 non-null  object        
 1   numbicicletas     100000 non-null  int64         
 2   numbicicletas__1  100000 non-null  int64         
 3   numdocasvacias    100000 non-null  int64         
 4   position          100000 non-null  object        
 5   entity_ts         100000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 4.6+ MB
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2218606 entries, 0 to 2218605
Data columns (total 7 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   desigcomercial  object
 1   numbicicletas   int64 
 2   numdocas        int64 
 3   numdocasvacias  int64 
 4   position        object
 5   entity_t

In [6]:
for name, df in files:
    print(f"{name:10} exact duplicate rows: {df.duplicated().sum():>7}")

2020_q1    exact duplicate rows:      46
2020_q2    exact duplicate rows:      58
2020_q3    exact duplicate rows:      46
2020_q4    exact duplicate rows:      18
2021       exact duplicate rows:  985748
2022_s1    exact duplicate rows:  673559
2022_s2    exact duplicate rows: 1147087
2023_q1    exact duplicate rows:  380169


In [7]:
for name, df in files:
    negs = {c: int((df[c] < 0).sum())
            for c in ['numbicicletas', 'numdocasvacias', 'numdocas']
            if c in df.columns and (df[c] < 0).any()}
    print(f"{name:10} negatives: {negs if negs else 'none'}")

2020_q1    negatives: {'numdocasvacias': 8}
2020_q2    negatives: {'numdocasvacias': 1}
2020_q3    negatives: {'numdocasvacias': 1}
2020_q4    negatives: none
2021       negatives: {'numdocasvacias': 86}
2022_s1    negatives: none
2022_s2    negatives: none
2023_q1    negatives: none


In [8]:
for name, df in files:
    if 'numdocas' in df.columns:
        n = int((df['numbicicletas'] > df['numdocas']).sum())
        print(f"{name:10} bikes > capacity: {n:>6}  ({n/len(df)*100:.3f}%)")

2021       bikes > capacity:     86  (0.004%)
2022_s1    bikes > capacity:     48  (0.003%)
2022_s2    bikes > capacity:     56  (0.002%)
2023_q1    bikes > capacity:     18  (0.002%)


In [9]:
lhs = gira_2021['numbicicletas'] + gira_2021['numdocasvacias']
mask = lhs == gira_2021['numdocas']
print(f"Rows passing bikes + empty == capacity: "
      f"{mask.sum():,} / {len(gira_2021):,} = {mask.mean()*100:.2f}%")

Rows passing bikes + empty == capacity: 2,218,606 / 2,218,606 = 100.00%


## Step 2 - Cleaning the Data

In [10]:
for name, df in [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
                 ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4)]:
    identical = (df['numbicicletas'] == df['numbicicletas__1']).all()
    print(f"{name}: numbicicletas__1 identical to numbicicletas? {identical}")

2020_q1: numbicicletas__1 identical to numbicicletas? True
2020_q2: numbicicletas__1 identical to numbicicletas? True
2020_q3: numbicicletas__1 identical to numbicicletas? True
2020_q4: numbicicletas__1 identical to numbicicletas? True


In [11]:
gira_2020_q1 = gira_2020_q1.drop(columns='numbicicletas__1')
gira_2020_q2 = gira_2020_q2.drop(columns='numbicicletas__1')
gira_2020_q3 = gira_2020_q3.drop(columns='numbicicletas__1')
gira_2020_q4 = gira_2020_q4.drop(columns='numbicicletas__1')

print(gira_2020_q1.columns.tolist())

['desigcomercial', 'numbicicletas', 'numdocasvacias', 'position', 'entity_ts']


In [12]:
files = [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
         ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4),
         ("2021", gira_2021), ("2022_s1", gira_2022_s1),
         ("2022_s2", gira_2022_s2), ("2023_q1", gira_2023_q1)]

In [13]:
for name, df in files:
    df['desigcomercial'] = df['desigcomercial'].str.strip()
print("Whitespace stripped from all 8 dfs.")

Whitespace stripped from all 8 dfs.


In [14]:
for name, df in files:
    mask = df['desigcomercial'].str.startswith('103') & \
           (df['desigcomercial'] != '103 - Jardim da Água')
    df.loc[mask, 'desigcomercial'] = '103 - Jardim da Água'

# verify: station 103 should now have exactly ONE name form
for name, df in files:
    names_103 = df.loc[df['desigcomercial'].str.startswith('103'), 'desigcomercial'].unique()
    if len(names_103) > 0:
        print(f"{name}: {list(names_103)}")

2020_q1: ['103 - Jardim da Água']
2020_q2: ['103 - Jardim da Água']
2020_q3: ['103 - Jardim da Água']
2020_q4: ['103 - Jardim da Água']
2021: ['103 - Jardim da Água']
2022_s1: ['103 - Jardim da Água']
2022_s2: ['103 - Jardim da Água']
2023_q1: ['103 - Jardim da Água']


In [15]:
for name, df in files:
    df['station_id'] = df['desigcomercial'].str.extract(r'^(\d+)').astype('Int64')

In [16]:
for name, df in files:
    n_missing = df['station_id'].isna().sum()
    n_stations = df['station_id'].nunique()
    print(f"{name:10} unique station_ids: {n_stations:>4}  |  rows with no id: {n_missing}")

2020_q1    unique station_ids:   82  |  rows with no id: 1198
2020_q2    unique station_ids:   83  |  rows with no id: 1149
2020_q3    unique station_ids:   83  |  rows with no id: 1117
2020_q4    unique station_ids:   84  |  rows with no id: 0
2021       unique station_ids:  102  |  rows with no id: 1
2022_s1    unique station_ids:  132  |  rows with no id: 0
2022_s2    unique station_ids:  147  |  rows with no id: 0
2023_q1    unique station_ids:  147  |  rows with no id: 0


In [17]:
for name, df in [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
                 ("2020_q3", gira_2020_q3), ("2021", gira_2021)]:
    bad = df.loc[df['station_id'].isna(), 'desigcomercial'].unique()
    print(f"=== {name} ({df['station_id'].isna().sum()} rows) ===")
    print(list(bad))
    print()

=== 2020_q1 (1198 rows) ===
['Avenida das Gaivotas - Rua dos Corvos']

=== 2020_q2 (1149 rows) ===
['Avenida das Gaivotas - Rua dos Corvos']

=== 2020_q3 (1117 rows) ===
['Avenida das Gaivotas - Rua dos Corvos']

=== 2021 (1 rows) ===
['VC 21']



In [18]:
for name, df in [("2021", gira_2021), ("2022_s1", gira_2022_s1),
                 ("2022_s2", gira_2022_s2), ("2023_q1", gira_2023_q1)]:
    hits = df.loc[df['desigcomercial'].str.contains('Gaivotas', case=False, na=False),
                  'desigcomercial'].unique()
    print(f"{name}: {list(hits)}")

2021: []
2022_s1: []
2022_s2: []
2023_q1: []


In [19]:
# 1) Does it really not exist post-2020? Check a looser term.
for name, df in [("2021", gira_2021), ("2022_s1", gira_2022_s1),
                 ("2022_s2", gira_2022_s2), ("2023_q1", gira_2023_q1)]:
    hits = df.loc[df['desigcomercial'].str.contains('Corvos|Gaivota', case=False, na=False),
                  'desigcomercial'].unique()
    print(f"{name}: {list(hits)}")

# 2) In 2020, is this name ALWAYS without a number, or does a numbered version also exist?
print("\n2020 name forms containing Gaivotas/Corvos:")
for name, df in [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
                 ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4)]:
    hits = df.loc[df['desigcomercial'].str.contains('Gaivota|Corvos', case=False, na=False),
                  'desigcomercial'].unique()
    print(f"{name}: {list(hits)}")

2021: []
2022_s1: []
2022_s2: []
2023_q1: []

2020 name forms containing Gaivotas/Corvos:
2020_q1: ['Avenida das Gaivotas - Rua dos Corvos']
2020_q2: ['Avenida das Gaivotas - Rua dos Corvos']
2020_q3: ['Avenida das Gaivotas - Rua dos Corvos']
2020_q4: []


In [20]:
GAIVOTAS = 'Avenida das Gaivotas - Rua dos Corvos'
for name, df in files:
    df.loc[df['desigcomercial'] == GAIVOTAS, 'station_id'] = 9999

for name, df in files:
    print(f"{name:10} rows with no id: {df['station_id'].isna().sum()}")

2020_q1    rows with no id: 0
2020_q2    rows with no id: 0
2020_q3    rows with no id: 0
2020_q4    rows with no id: 0
2021       rows with no id: 1
2022_s1    rows with no id: 0
2022_s2    rows with no id: 0
2023_q1    rows with no id: 0


In [21]:
for name, df in [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
                 ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4)]:
    df['numdocas'] = df['numbicicletas'] + df['numdocasvacias']
    df['estado'] = pd.NA   # 2020 has no status column; keep it explicit, not invented

In [22]:
for name, df in [("2022_s1", gira_2022_s1), ("2022_s2", gira_2022_s2),
                 ("2023_q1", gira_2023_q1)]:
    df['numdocasvacias'] = df['numdocas'] - df['numbicicletas']

In [23]:
for name, df in files:
    cols = set(df.columns)
    needed = {'desigcomercial','numbicicletas','numdocas','numdocasvacias',
              'position','entity_ts','estado','station_id'}
    print(f"{name:10} has all core cols: {needed.issubset(cols)}  | extra: {cols - needed}")

2020_q1    has all core cols: True  | extra: set()
2020_q2    has all core cols: True  | extra: set()
2020_q3    has all core cols: True  | extra: set()
2020_q4    has all core cols: True  | extra: set()
2021       has all core cols: True  | extra: set()
2022_s1    has all core cols: True  | extra: set()
2022_s2    has all core cols: True  | extra: set()
2023_q1    has all core cols: True  | extra: set()


In [24]:
for name, df in [("2020_q1", gira_2020_q1), ("2020_q2", gira_2020_q2),
                 ("2020_q3", gira_2020_q3), ("2020_q4", gira_2020_q4)]:
    df['ts_local'] = df['entity_ts'].dt.tz_localize(
        'Europe/Lisbon', ambiguous='NaT', nonexistent='NaT')

In [25]:
for name, df in [("2021", gira_2021), ("2022_s1", gira_2022_s1),
                 ("2022_s2", gira_2022_s2), ("2023_q1", gira_2023_q1)]:
    df['ts_local'] = (pd.to_datetime(df['entity_ts'], utc=True, format='ISO8601')
                        .dt.tz_convert('Europe/Lisbon'))

In [26]:
for name, df in files:
    print(f"{name:10} dtype={str(df['ts_local'].dtype):30} "
          f"sample={df['ts_local'].iloc[0]}  NaT={df['ts_local'].isna().sum()}")

2020_q1    dtype=datetime64[ns, Europe/Lisbon]  sample=2020-01-01 00:04:01+00:00  NaT=70
2020_q2    dtype=datetime64[ns, Europe/Lisbon]  sample=2020-04-01 00:06:04+01:00  NaT=0
2020_q3    dtype=datetime64[ns, Europe/Lisbon]  sample=2020-06-30 00:06:06+01:00  NaT=0
2020_q4    dtype=datetime64[ns, Europe/Lisbon]  sample=2020-10-01 00:16:10+01:00  NaT=76
2021       dtype=datetime64[ns, Europe/Lisbon]  sample=2021-05-28 20:16:33.945000+01:00  NaT=0
2022_s1    dtype=datetime64[ns, Europe/Lisbon]  sample=2022-01-01 13:38:03.130000+00:00  NaT=0
2022_s2    dtype=datetime64[ns, Europe/Lisbon]  sample=2022-07-27 16:53:45.206000+01:00  NaT=0
2023_q1    dtype=datetime64[ns, Europe/Lisbon]  sample=2023-01-01 12:14:19+00:00  NaT=0


In [27]:
for name, df in [("2020_q1", gira_2020_q1), ("2020_q4", gira_2020_q4)]:
    bad = df[df['ts_local'].isna()]
    print(f"=== {name}: {len(bad)} NaT rows ===")
    print("  original entity_ts range of NaT rows:")
    print("   ", bad['entity_ts'].min(), "->", bad['entity_ts'].max())
    print("   unique dates:", sorted(bad['entity_ts'].dt.date.unique()))

=== 2020_q1: 70 NaT rows ===
  original entity_ts range of NaT rows:
    2020-03-29 01:00:03 -> 2020-03-29 01:40:03
   unique dates: [datetime.date(2020, 3, 29)]
=== 2020_q4: 76 NaT rows ===
  original entity_ts range of NaT rows:
    2020-10-25 01:06:10 -> 2020-10-25 01:46:10
   unique dates: [datetime.date(2020, 10, 25)]


In [28]:
gira = pd.concat(
    [gira_2020_q1, gira_2020_q2, gira_2020_q3, gira_2020_q4,
     gira_2021, gira_2022_s1, gira_2022_s2, gira_2023_q1],
    ignore_index=True
)
print("Combined shape:", gira.shape)
print("Total expected:", 100000*4 + 2218606 + 1555396 + 2382896 + 933644)
print("\nDtypes:\n", gira.dtypes)
print("\nRows per source year (via ts_local):")
print(gira['ts_local'].dt.year.value_counts(dropna=False).sort_index())

Combined shape: (7490542, 9)
Total expected: 7490542

Dtypes:
 desigcomercial                           object
numbicicletas                             int64
numdocasvacias                            int64
position                                 object
entity_ts                                object
station_id                                Int64
numdocas                                  int64
estado                                   object
ts_local          datetime64[ns, Europe/Lisbon]
dtype: object

Rows per source year (via ts_local):
ts_local
2020.0     399854
2021.0    2218606
2022.0    3348370
2023.0    1523566
NaN           146
Name: count, dtype: int64


In [29]:
# Drop the single non-station row (VC 21 in 2021) whose name has no numeric
# prefix, so station_id is <NA>. It is not a real GIRA station. We remove it
# BEFORE deduplication, because Pass 2 dedups on (station_id, entity_ts) and a
# stray <NA> id could otherwise collapse unrelated rows sharing a timestamp.
before = len(gira)
gira = gira[gira['station_id'].notna()].copy()
print(f"Removed {before - len(gira)} row(s) with <NA> station_id")

assert gira['station_id'].isna().sum() == 0, \
    "Unexpected unparseable station names remain — investigate before dedup"
print("station_id fully clean:", gira['station_id'].isna().sum() == 0)

Removed 1 row(s) with <NA> station_id
station_id fully clean: True


In [30]:
n_start = len(gira)
print(f"Starting rows: {n_start:,}")

# Pass 1 — fully identical rows
gira = gira.drop_duplicates()
n_after_p1 = len(gira)
print(f"Pass 1 (exact dupes)   removed: {n_start - n_after_p1:>10,}  ->  {n_after_p1:,} rows")

# Pass 2 — same station + timestamp, keep first
gira = gira.drop_duplicates(subset=['station_id', 'entity_ts'], keep='first')
n_after_p2 = len(gira)
print(f"Pass 2 (station+ts)    removed: {n_after_p1 - n_after_p2:>10,}  ->  {n_after_p2:,} rows")

print(f"\nTotal removed: {n_start - n_after_p2:,} ({(n_start - n_after_p2)/n_start*100:.1f}%)")
print(f"Final rows: {n_after_p2:,}")

Starting rows: 7,490,541
Pass 1 (exact dupes)   removed:  3,398,561  ->  4,091,980 rows
Pass 2 (station+ts)    removed:      1,801  ->  4,090,179 rows

Total removed: 3,400,362 (45.4%)
Final rows: 4,090,179


In [31]:
print("Rows per year after dedup:")
print(gira['ts_local'].dt.year.value_counts(dropna=False).sort_index())
print(f"\nTotal: {len(gira):,}")

Rows per year after dedup:
ts_local
2020.0     398163
2021.0    1232857
2022.0    1852486
2023.0     606527
NaN           146
Name: count, dtype: int64

Total: 4,090,179


In [32]:
gira['numdocasvacias_fix'] = gira['numdocasvacias'].clip(lower=0)

changed = (gira['numdocasvacias_fix'] != gira['numdocasvacias']).sum()
print(f"Values changed by clip: {changed}")
print(gira.loc[gira['numdocasvacias'] < 0,
               ['numdocasvacias', 'numdocasvacias_fix']].head())

Values changed by clip: 116
       numdocasvacias  numdocasvacias_fix
14228              -1                   0
18828              -1                   0
27862              -1                   0
28629              -1                   0
37926              -1                   0


In [33]:
gira['is_impossible'] = gira['numbicicletas'] > gira['numdocas']
print(f"Impossible rows flagged: {gira['is_impossible'].sum():,} "
      f"({gira['is_impossible'].mean()*100:.4f}%)")

Impossible rows flagged: 116 (0.0028%)


In [34]:
gira['is_repair'] = gira['estado'] == 'repair'
print(f"Repair rows flagged: {gira['is_repair'].sum():,} "
      f"({gira['is_repair'].mean()*100:.2f}%)")
print("\nestado value counts:")
print(gira['estado'].value_counts(dropna=False))

Repair rows flagged: 160,208 (3.92%)

estado value counts:
estado
active    3531662
NaN        398309
repair     160208
Name: count, dtype: int64


In [35]:
# Extract the two numbers inside "coordinates": [lon, lat]
coords = gira['position'].str.extract(
    r'\[\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\]'
)
gira['lon'] = pd.to_numeric(coords[0], errors='coerce')  # first = longitude
gira['lat'] = pd.to_numeric(coords[1], errors='coerce')  # second = latitude

# verify
print("Parse failures (NaN):")
print(f"  lon: {gira['lon'].isna().sum()},  lat: {gira['lat'].isna().sum()}")
print("\nCoordinate ranges:")
print(f"  lon: {gira['lon'].min():.5f}  to  {gira['lon'].max():.5f}")
print(f"  lat: {gira['lat'].min():.5f}  to  {gira['lat'].max():.5f}")

Parse failures (NaN):
  lon: 0,  lat: 0

Coordinate ranges:
  lon: -9.22270  to  0.00000
  lat: 0.00000  to  38.79300


In [36]:
# How many rows have a zero coordinate?
zero_mask = (gira['lat'] == 0) | (gira['lon'] == 0)
print(f"Rows with a zero coordinate: {zero_mask.sum():,}")

# Which stations, and do they have GOOD coords elsewhere?
zero_stations = gira.loc[zero_mask, 'station_id'].value_counts()
print(f"\nStations affected: {len(zero_stations)}")
print(zero_stations.head(20))

# Look at the raw position text for a few of these
print("\nRaw position values that produced a zero:")
print(gira.loc[zero_mask, 'position'].value_counts().head())

Rows with a zero coordinate: 1

Stations affected: 1
station_id
113    1
Name: count, dtype: Int64

Raw position values that produced a zero:
position
{'coordinates': [0.0, 0.0], 'type': 'Point'}    1
Name: count, dtype: int64


In [37]:
# Find station 113's correct coordinates from its valid rows
good_113 = gira[(gira['station_id'] == 113) & (gira['lat'] != 0) & (gira['lon'] != 0)]
true_lat = good_113['lat'].median()
true_lon = good_113['lon'].median()
print(f"Station 113 true location: lat={true_lat:.5f}, lon={true_lon:.5f}")

# Backfill the single zero row
zero_mask = (gira['lat'] == 0) | (gira['lon'] == 0)
gira.loc[zero_mask, 'lat'] = true_lat
gira.loc[zero_mask, 'lon'] = true_lon

# re-verify the ranges are now clean
print(f"\nAfter fix:")
print(f"  lon: {gira['lon'].min():.5f}  to  {gira['lon'].max():.5f}")
print(f"  lat: {gira['lat'].min():.5f}  to  {gira['lat'].max():.5f}")
print(f"  zero coords remaining: {((gira['lat']==0)|(gira['lon']==0)).sum()}")

Station 113 true location: lat=38.78414, lon=-9.09825

After fix:
  lon: -9.22270  to  -9.09235
  lat: 38.69350  to  38.79300
  zero coords remaining: 0


In [38]:
gira['occupancy_rate'] = gira['numbicicletas'] / gira['numdocas']

In [39]:
valid = ~gira['is_repair'] & ~gira['is_impossible']

gira['is_empty'] = valid & (gira['numbicicletas'] == 0)
gira['is_full']  = valid & (gira['numdocasvacias_fix'] == 0)

print(f"is_empty: {gira['is_empty'].sum():,} ({gira['is_empty'].mean()*100:.2f}% of all rows)")
print(f"is_full:  {gira['is_full'].sum():,} ({gira['is_full'].mean()*100:.2f}% of all rows)")
print(f"\nBoth true at once (should be rare — only if capacity is 0): "
      f"{(gira['is_empty'] & gira['is_full']).sum()}")

is_empty: 206,000 (5.04% of all rows)
is_full:  80,166 (1.96% of all rows)

Both true at once (should be rare — only if capacity is 0): 2


In [40]:
both = gira['is_empty'] & gira['is_full']
print(gira.loc[both, ['station_id', 'numbicicletas', 'numdocas',
                      'numdocasvacias_fix', 'estado', 'ts_local']])
print("\nnumdocas value counts near zero:")
print((gira['numdocas'] == 0).sum(), "rows have numdocas == 0")

         station_id  numbicicletas  numdocas  numdocasvacias_fix  estado  \
2364585         133              0         0                   0  active   
3696883         139              0         0                   0  active   

                                ts_local  
2364585 2021-07-27 20:43:08.473000+01:00  
3696883 2022-02-03 11:37:07.276000+00:00  

numdocas value counts near zero:
20860 rows have numdocas == 0


In [41]:
zc = gira['numdocas'] == 0
print(f"Zero-capacity rows: {zc.sum():,} ({zc.mean()*100:.3f}%)")
print(f"\nStations affected: {gira.loc[zc, 'station_id'].nunique()}")
print("\nTop stations by zero-capacity row count:")
print(gira.loc[zc, 'station_id'].value_counts().head(10))
print("\nBy year:")
print(gira.loc[zc, 'ts_local'].dt.year.value_counts().sort_index())
print("\nDo these stations have GOOD (non-zero) capacity rows elsewhere?")
for sid in gira.loc[zc, 'station_id'].value_counts().head(3).index:
    good = gira[(gira['station_id']==sid) & (gira['numdocas']>0)]
    print(f"  station {sid}: {len(good):,} good rows, "
          f"typical capacity={good['numdocas'].median() if len(good) else 'NONE'}")

Zero-capacity rows: 20,860 (0.510%)

Stations affected: 13

Top stations by zero-capacity row count:
station_id
238    14505
508      757
232      744
361      742
510      739
518      729
365      724
517      715
362      602
210      599
Name: count, dtype: Int64

By year:
ts_local
2021        1
2022    16784
2023     4075
Name: count, dtype: int64

Do these stations have GOOD (non-zero) capacity rows elsewhere?
  station 238: 14,440 good rows, typical capacity=36.0
  station 508: 14,458 good rows, typical capacity=23.0
  station 232: 14,488 good rows, typical capacity=22.0


In [42]:
# Fold zero-capacity into the impossible flag
gira['is_impossible'] = (gira['numbicicletas'] > gira['numdocas']) | (gira['numdocas'] == 0)
print(f"is_impossible now flags: {gira['is_impossible'].sum():,} "
      f"({gira['is_impossible'].mean()*100:.3f}%)")

# Recompute the failure flags so they respect the updated impossible flag
valid = ~gira['is_repair'] & ~gira['is_impossible']
gira['is_empty'] = valid & (gira['numbicicletas'] == 0)
gira['is_full']  = valid & (gira['numdocasvacias_fix'] == 0)

print(f"\nRecomputed:")
print(f"  is_empty: {gira['is_empty'].sum():,} ({gira['is_empty'].mean()*100:.2f}%)")
print(f"  is_full:  {gira['is_full'].sum():,} ({gira['is_full'].mean()*100:.2f}%)")
print(f"  both at once: {(gira['is_empty'] & gira['is_full']).sum()}")

# occupancy_rate was computed in an earlier cell as bikes / numdocas,
# BEFORE zero-capacity rows were folded into is_impossible above. Those rows
# hold inf (bikes/0) or NaN (0/0) and would poison any mean of occupancy_rate.
# Null them out now that is_impossible is final, so aggregates are safe.
gira.loc[gira['is_impossible'], 'occupancy_rate'] = pd.NA
inf_left = gira['occupancy_rate'].isin([float('inf'), float('-inf')]).sum()
print(f"\ninf values in occupancy_rate: {inf_left}")
print(f"occupancy_rate range: {gira['occupancy_rate'].min():.3f} "
      f"to {gira['occupancy_rate'].max():.3f}")


is_impossible now flags: 20,976 (0.513%)

Recomputed:
  is_empty: 205,998 (5.04%)
  is_full:  80,164 (1.96%)
  both at once: 0

inf values in occupancy_rate: 0
occupancy_rate range: 0.000 to 1.000


In [43]:
gira['year']    = gira['ts_local'].dt.year
gira['quarter'] = gira['ts_local'].dt.quarter
gira['month']   = gira['ts_local'].dt.month
gira['date']    = gira['ts_local'].dt.date
gira['hour']    = gira['ts_local'].dt.hour
gira['weekday'] = gira['ts_local'].dt.dayofweek   # Monday=0 ... Sunday=6

In [44]:
def assign_period(ts):
    if pd.isna(ts):
        return pd.NA
    d = ts.date()
    y = ts.year
    if y == 2020:
        import datetime as dt
        if d < dt.date(2020, 3, 18):
            return 'Pre-COVID'
        elif d <= dt.date(2020, 5, 3):      # Portugal's state of emergency window
            return 'Lockdown'
        else:
            return 'Reopening'
    else:
        return str(y)   # '2021', '2022', '2023'

gira['period'] = gira['ts_local'].apply(assign_period)

print(gira['period'].value_counts(dropna=False))

period
2022         1852486
2021         1232857
2023          606527
Reopening     248870
Pre-COVID      81131
Lockdown       68162
<NA>             146
Name: count, dtype: int64


In [45]:
print(f"Rows: {len(gira):,}")
print(f"Memory: {gira.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nDtypes:")
print(gira.dtypes)

Rows: 4,090,179
Memory: 2309.2 MB

Dtypes:
desigcomercial                               object
numbicicletas                                 int64
numdocasvacias                                int64
position                                     object
entity_ts                                    object
station_id                                    Int64
numdocas                                      int64
estado                                       object
ts_local              datetime64[ns, Europe/Lisbon]
numdocasvacias_fix                            int64
is_impossible                                  bool
is_repair                                      bool
lon                                         float64
lat                                         float64
occupancy_rate                              float64
is_empty                                       bool
is_full                                        bool
year                                        float64
quarter              

In [46]:
# Counts: tiny integers (capacity maxes ~40) -> smallest int that fits
for c in ['numbicicletas', 'numdocas', 'numdocasvacias', 'numdocasvacias_fix']:
    gira[c] = pd.to_numeric(gira[c], downcast='integer')

# station_id: nullable, max 9999 -> Int16 (keeps the one <NA>)
gira['station_id'] = gira['station_id'].astype('Int16')

# Time parts: have <NA> on the 146 DST rows -> nullable ints
gira['year']    = gira['year'].astype('Int16')
gira['quarter'] = gira['quarter'].astype('Int8')
gira['month']   = gira['month'].astype('Int8')
gira['hour']    = gira['hour'].astype('Int8')
gira['weekday'] = gira['weekday'].astype('Int8')

# occupancy_rate: a ratio, float32 is plenty
gira['occupancy_rate'] = gira['occupancy_rate'].astype('float32')

# Low-cardinality text -> category (the big memory win: ~150 names, 2 states, 6 periods)
for c in ['desigcomercial', 'estado', 'period']:
    gira[c] = gira[c].astype('category')

# lat/lon deliberately left as float64
print("Downcast done.")
print(f"Memory now: {gira.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Downcast done.
Memory now: 1146.8 MB


In [47]:
gira_final = gira.drop(columns=['position']).copy()
gira_final['entity_ts'] = gira_final['entity_ts'].astype(str)

gira_final.to_parquet('gira_clean.parquet', index=False)
print("Saved gira_clean.parquet")

Saved gira_clean.parquet


In [48]:
check = pd.read_parquet('gira_clean.parquet')
print(f"File size: {os.path.getsize('gira_clean.parquet') / 1e6:.1f} MB")
print(f"Rows match: {len(check) == len(gira_final)}  ({len(check):,})")
print(f"Empty rate: {check['is_empty'].mean()*100:.2f}%")
print(f"Full rate:  {check['is_full'].mean()*100:.2f}%")
print(f"ts_local tz-aware: {check['ts_local'].dtype}")
print(f"lat/lon float64: {check['lat'].dtype}, {check['lon'].dtype}")

File size: 85.3 MB
Rows match: True  (4,090,179)
Empty rate: 5.04%
Full rate:  1.96%
ts_local tz-aware: datetime64[ns, Europe/Lisbon]
lat/lon float64: float64, float64


In [49]:
gira_final.to_csv('gira_clean.csv', index=False, encoding='utf-8-sig')
print(f"gira_clean.csv re-saved with BOM: {os.path.getsize('gira_clean.csv') / 1e6:.0f} MB")

gira_clean.csv re-saved with BOM: 805 MB
